# Baby-Chalk (1.5B) RL Post-Training on Tesla T4 (Google Colab)

Runs the full post-training stack on a free Google Colab Tesla T4 GPU (16 GB):
- **Cursor-Style Modified GRPO Loss**: unscaled mean-centered advantages, no token length normalization, and detached CISPO ratio clipping.
- **Rottweiler Verifier**: 5-stage XML scaffold (`<explore>`, `<conjecture>`, `<test_edge_cases>`, `<lemma_isolate>`, `<formal_proof>`) and SymPy symbolic reverse substitution.
- **Calibrated Abstention**: Asymmetric reward schedule (+1.0 correct, 0.0 abstain, -1.5 hallucinated/incorrect).
- **Anti-Looping Control**: 4-gram repetition stopping criteria with a 256-token sliding window.
- **Format Replay Guard**: 15–20% general conversation replay penalizing leaked reasoning tags (-1.5 penalty).
- **Memory Architecture**: PyTorch SDPA memory-efficient attention and PEFT LoRA, fitting rollouts comfortably in ~12 GB VRAM.

In [ ]:
# 1. Verify Tesla T4 GPU Hardware
!nvidia-smi

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")


In [ ]:
# 2. Clone Repository and Install Dependencies
import os

# Always reset to root /content first to prevent nested directory traps
%cd /content

# Clean up accidental nested clone if it occurred
!rm -rf /content/t4-cuda/t4-cuda

if not os.path.exists("/content/t4-cuda"):
    !git clone -b feat-big-chalk-math-7b https://github.com/Epoch-AI-Lab/t4-cuda.git /content/t4-cuda

%cd /content/t4-cuda
!git pull
!pwd

!pip install -q transformers peft accelerate sympy datasets


In [ ]:
# 3. Mount Google Drive for Persistent Checkpoint Storage (Recommended)
from google.colab import drive
import os

USE_DRIVE = True  # Set to False if training purely in local ephemeral storage
OUTPUT_DIR = "results/baby_chalk_rl"

if USE_DRIVE:
    try:
        drive.mount("/content/drive")
        OUTPUT_DIR = "/content/drive/MyDrive/baby_chalk_rl_checkpoints"
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        print(f"Checkpoints will be saved to Google Drive: {OUTPUT_DIR}")
    except Exception as e:
        print("Drive mount skipped or failed, using local directory:", OUTPUT_DIR)


In [ ]:
# 4. Cold-Start SFT (Phase 1: ~1-2 mins on Tesla T4)
# Teaches the raw Qwen2.5-Math-1.5B model the 5-stage XML scaffold (<explore>, <conjecture>, etc.)
!python3 benchmarks/train_math_sft.py \
    --model_name_or_path "Qwen/Qwen2.5-Math-1.5B" \
    --data_path "data/chalk_seeds_500.jsonl" \
    --output_dir "results/chalk_math_1.5b_sft" \
    --epochs 1 \
    --max_length 1024 \
    --batch_size 2 \
    --grad_accum 8 \
    --lora_r 16 \
    --lora_alpha 32


In [ ]:
# 5. Launch Baby-Chalk (1.5B) RL Post-Training Run (Phase 2)
# Loads the SFT adapter from Phase 1 and runs GRPO with Rottweiler verification
!python3 benchmarks/train_baby_chalk_rl.py \
    --model_name_or_path "Qwen/Qwen2.5-Math-1.5B" \
    --sft_adapter_path "results/chalk_math_1.5b_sft/lora_adapter" \
    --data_path "data/chalk_seeds_500.jsonl" \
    --output_dir {OUTPUT_DIR} \
    --max_steps 30 \
    --batch_size 1 \
    --num_generations 4 \
    --learning_rate 2e-5 \
    --max_prompt_len 512 \
    --max_completion_len 512 \
    --temperature 0.8 \
    --lora_r 16 \
    --lora_alpha 32 \
    --save_steps 10


In [ ]:
# 6. Evaluate Post-Trained Checkpoint on Held-Out AMC / AIME Competition Problems
!python3 benchmarks/eval_math_benchmark.py \
    --model_path "Qwen/Qwen2.5-Math-1.5B" \
    --lora_path f"{OUTPUT_DIR}/checkpoint_step_30" \
    --eval_path "data/external_math_eval.json" \
    --max_new_tokens 1024
